# Test notebook

The purpose of this notebook is to test an equation and compare them with the baselines: Burton, MBR, and DDM1, 2 and 3. We will also plot each storm and get the metrics for the equation.

The only cell that we have to modify is the following one, where we can change the features, the mode (template or default) and the output directory for the plots.
Raw EQ is the equation that we want to test, the raw version generated from the train_script.py file.

In [2]:
import os

FEATURES = ["Vp", "Np", "Bzsouth", "Bmag", "DST"]
MODE = "template"  # 'template' or 'default'
OUTPUT_DIR = "template_primitive_features"
RAW_EQS = [
    'g = (#3 - -2.152752) * ((((4.7333016 - #2) * #3) - #1) / 501.94562); d = square(1.4046433 - (#1 * 0.015329162))',
    'g = ((#1 + (#3 * (#2 - sqrt(#3)))) * 0.0020573507) * (-1.775414 - #3); d = square((#1 * -0.015831897) + 1.3089455)',
    'g = (#3 - -2.1569605) * (((#2 * #3) + #1) * -0.0018313164); d = square((#1 * -0.015403083) + 1.3434031)',
    'g = (#2 + (#1 * ((#3 * (-0.301232 - ((#1 * 3.5000103e-5) * #2))) - 0.55653155))) * 0.0053468808; d = square((0.015961226 * #1) + -1.2015269)',
    'g = (((#3 * (#2 - sqrt(#4))) + #1) * (1.6439178 + #3)) * -0.0021015273; d = square(-1.2798462 - (#1 * -0.015890777))',
    'g = ((#1 + (#2 * #3)) * 0.0018150203) * (-2.0452397 - #3); d = square((#1 * -0.015376654) - -1.3081883)',
    'g = ((#2 * ((#1 * #3) * 0.00011782655)) + (#3 + 1.3935444)) * (#1 * -0.0016007437); d = square((#1 * -0.016486706) - -1.0748392)',
]

output_folder = OUTPUT_DIR
# Count number of existing subfolders
if not os.path.exists(output_folder):
    os.makedirs(output_folder)

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import MultipleLocator

import sympy as sp
from tqdm import tqdm

from sympy.printing import latex

# Internal module imports
import storm_dates
import baseline_models

# from evaluation_engine import UnifiedModel, simulate_storm, compute_features
from evaluation_engine import EquationModel, simulate_storm
from train_script import load_and_preprocess, compute_features

Detected IPython. Loading juliacall extension. See https://juliapy.github.io/PythonCall.jl/stable/compat/#IPython


In [4]:
raw_data = load_and_preprocess()
data = compute_features(raw_data)

Reading from file ./data/all_timeline/ace_imf_1h_1998.csv
Reading from file ./data/all_timeline/ace_imf_1h_1999.csv
Reading from file ./data/all_timeline/ace_imf_1h_2000.csv
Reading from file ./data/all_timeline/ace_imf_1h_2001.csv
Reading from file ./data/all_timeline/ace_imf_1h_2002.csv
Reading from file ./data/all_timeline/ace_imf_1h_2003.csv
Reading from file ./data/all_timeline/ace_imf_1h_2004.csv
Reading from file ./data/all_timeline/ace_imf_1h_2005.csv
Reading from file ./data/all_timeline/ace_imf_1h_2006.csv
Reading from file ./data/all_timeline/ace_imf_1h_2007.csv
Reading from file ./data/all_timeline/ace_imf_1h_2008.csv
Reading from file ./data/all_timeline/ace_imf_1h_2009.csv
Reading from file ./data/all_timeline/ace_imf_1h_2010.csv
Reading from file ./data/all_timeline/ace_imf_1h_2011.csv
Reading from file ./data/all_timeline/ace_imf_1h_2012.csv
Reading from file ./data/all_timeline/ace_imf_1h_2013.csv
Reading from file ./data/all_timeline/ace_imf_1h_2014.csv
Reading from f

In [ ]:
def predict_and_plot_storm(model, eqs, start, end, storm_df, storm_id, save_path):
    # 1. Generate Predictions
    y_true = storm_df[start:end]["DST"].values
    colors = ["blue", "yellow", "green", "orange", "purple", "cyan", "magenta"]
    res_eqs = []
    string_title = f"Storm {storm_id} Reconstruction\n"
    metrics_info = []
    
    
    for eq_index, eq in enumerate(eqs):
        res_eq = simulate_storm(model[eq], storm_df)
        res_eq = res_eq[start:end]["DST_pred"].values
        m_eq = baseline_models.get_all_metrics_dict(y_true, res_eq)
        metrics_info.append(m_eq)
        string_title += f"Evaluation for Equation {eq_index + 1} ({colors[eq_index]}): ${model[eq].latex_str()}$ \n"
    
        res_eqs.append(res_eq)

    # 2. Calculate Metrics
    
    # 3. Setup Figure (3 Columns)
    fig, axs = plt.subplots(1, 3, figsize=(24, 7), constrained_layout=True)
    fig.suptitle(
        string_title, fontsize=18
    )
    # Column 1: Time Series
    axs[0].plot(
        storm_df[start:end].index,
        y_true,
        color="black",
        label="Observed",
        alpha=0.6,
        linewidth=2,
    )
    
    for eq_index, res_eq in enumerate(res_eqs):
    
        axs[0].plot(
            storm_df[start:end].index,
            res_eqs[eq_index],
            color=colors[eq_index],
            linestyle="--",
            label=f"Equation {eq_index + 1}",
            linewidth=1.5,
        )
    
    axs[0].tick_params(axis='both', which='major', labelsize=14)
    axs[0].tick_params(axis='both', which='minor', labelsize=10)
    axs[0].legend(fontsize = 16)
    axs[0].grid(True)
    axs[0].set_xlim(start, end)
    

    axs[0].xaxis.set_major_locator(MultipleLocator(2))
    
    diffs = []
    
    for eq_index, res_eq in enumerate(res_eqs):
        diff_eq = res_eq - y_true
        diffs.append(diff_eq)

        axs[1].plot(storm_df[start:end].index, diff_eq, color=colors[eq_index], label=f"Equation {eq_index + 1} Error")
    
    axs[1].axhline(0, color="black", linestyle="--")

    title_metrics = (
        f"Error Comparison\n"       
    )
    
    for eq_index, m_eq in enumerate(metrics_info):
        title_metrics += f"Eq {eq_index + 1}: MAE={m_eq['MAE']:.2f}, RMSE={m_eq['RMSE']:.2f}, R²={m_eq['R2']:.3f}, BFE={m_eq['BFE']:.3f}\n"
        
        
    axs[1].set_title(title_metrics, fontsize=18)
    axs[1].set_ylabel("Error (nT)", fontsize=16)
    axs[1].legend(fontsize=16)
    axs[1].grid(True)
    axs[1].set_xlim(start, end)
    axs[1].tick_params(axis='both', which='major', labelsize=14)
    axs[1].tick_params(axis='both', which='minor', labelsize=10)
    
    axs[1].xaxis.set_major_locator(MultipleLocator(2))

    # Column 3: BFE
    baseline_models.plot_evaluation_bfe_multi(
        axs[2],
        y_true,
        res_eqs,
        [f"Equation {i+1}" for i in range(len(eqs))],
        [colors[i] for i in range(len(eqs))],
        fontsize = 16   
    )

    plt.savefig(save_path)
    plt.close()


In [6]:
def save_prediction_data(model, eqs, start, end, storm_df, output_path):
    """
    Generates and saves a CSV with observed and predicted DST and dDST/dt.
    """
    # 1. Observed Data
    # Real dDST is calculated as the difference to the next hour
    real_dst = storm_df[start:end]["DST"].values
    real_ddst = storm_df[start:end]["DST"].diff().shift(-1).values

    # 2. Equation Predictions
    # We need the iterative predictions for DST
    pred_dst_eqs = []
    for eq_index, eq in enumerate(eqs):
        pred_dst_eq = simulate_storm(model[eq], storm_df)
        if model[eq].is_template:
            pred_dst_eq = pred_dst_eq[start:end][
                ["DST_pred", "dDST", "injection_component", "decay_component"]
            ]
        else:
            pred_dst_eq = pred_dst_eq[start:end][["DST_pred", "dDST"]]
        pred_dst_eqs.append(pred_dst_eq)
        
    # 3. Baseline Predictions (Burton & OBM)
    
    # 4. Construct Comprehensive DataFrame

    if model[eq].is_template:
        results_df = pd.DataFrame(
            {
                "Timestamp": storm_df[start:end].index,
                "Observed_DST": real_dst,
                "Real_dDST_dt": real_ddst,
                "Pred_DST_Equation": pred_dst_eq["DST_pred"].values,
                "Pred_dDST_dt_Equation": pred_dst_eq["dDST"].values,
                "Injection_Component": pred_dst_eq["injection_component"].values,
                "Decay_Component": pred_dst_eq["decay_component"].values,
                
            }
        ).set_index("Timestamp")
        
        for eq_index, pred_dst_eq in enumerate(pred_dst_eqs):
            results_df[f"Pred_DST_Equation_{eq_index+1}"] = pred_dst_eq["DST_pred"].values
            results_df[f"Pred_dDST_dt_Equation_{eq_index+1}"] = pred_dst_eq["dDST"].values
            results_df[f"Injection_Component_{eq_index+1}"] = pred_dst_eq["injection_component"].values
            results_df[f"Decay_Component_{eq_index+1}"] = pred_dst_eq["decay_component"].values
        
    else:
        results_df = pd.DataFrame(
            {
                "Timestamp": storm_df[start:end].index,
                "Observed_DST": real_dst,
                "Real_dDST_dt": real_ddst,                
            }
        ).set_index("Timestamp")
        
        for eq_index, pred_dst_eq in enumerate(pred_dst_eqs):
            results_df[f"Pred_DST_Equation_{eq_index+1}"] = pred_dst_eq["DST_pred"].values
            results_df[f"Pred_dDST_dt_Equation_{eq_index+1}"] = pred_dst_eq["dDST"].values


    results_df.to_csv(output_path)
    return results_df

## Test storms

In [7]:
storms = []
storm_indices = []
models = {}

for RAW_EQ in RAW_EQS:
    model = EquationModel(RAW_EQ, FEATURES, is_template=MODE == "template")
    models[RAW_EQ] = model
    
    
test_storms = storm_dates.TEST_STORMS_SYMBOLIC_REGRESSION

for sd, ed, storm_id in tqdm(test_storms):
    start = pd.to_datetime(sd)
    end = pd.to_datetime(ed)
    storm_df = data[
        start - pd.DateOffset(hours=1) : end + pd.DateOffset(hours=1)
    ].copy()
    if storm_df.empty:
        continue

    file_name = f"storm_{storm_id}.png"
    predict_and_plot_storm(
        models,
        RAW_EQS,
        start,
        end,
        storm_df,
        storm_id,
        os.path.join(OUTPUT_DIR, file_name),
    )

    csv_name = f"data_storm_{storm_id}.csv"
    storms.append(
        save_prediction_data(
            models, RAW_EQS, start, end, storm_df, os.path.join(OUTPUT_DIR, csv_name)
        )
    )
    storm_indices.append(storm_id)
    
with open(os.path.join(OUTPUT_DIR, 'equation.txt'), 'a') as f:
    f.write(f'Equation: {RAW_EQ}\n')            
    f.write(f'LaTeX: {latex(models[RAW_EQ].latex_str())}\n')

  0%|          | 0/20 [00:00<?, ?it/s]

  0%|          | 0/20 [00:00<?, ?it/s]


IndexError: list index out of range

In [ ]:
storms[0].columns

Index(['Observed_DST', 'Real_dDST_dt', 'Pred_DST_Equation',
       'Pred_dDST_dt_Equation', 'Injection_Component', 'Decay_Component',
       'Pred_DST_Equation_1', 'Pred_dDST_dt_Equation_1',
       'Injection_Component_1', 'Decay_Component_1', 'Pred_DST_Equation_2',
       'Pred_dDST_dt_Equation_2', 'Injection_Component_2', 'Decay_Component_2',
       'Pred_DST_Equation_3', 'Pred_dDST_dt_Equation_3',
       'Injection_Component_3', 'Decay_Component_3', 'Pred_DST_Equation_4',
       'Pred_dDST_dt_Equation_4', 'Injection_Component_4', 'Decay_Component_4',
       'Pred_DST_Equation_5', 'Pred_dDST_dt_Equation_5',
       'Injection_Component_5', 'Decay_Component_5', 'Pred_DST_Equation_6',
       'Pred_dDST_dt_Equation_6', 'Injection_Component_6', 'Decay_Component_6',
       'Pred_DST_Equation_7', 'Pred_dDST_dt_Equation_7',
       'Injection_Component_7', 'Decay_Component_7'],
      dtype='object')

In [ ]:
metrics = ["RMSE", "MAE", "R2", "CC", "BFE"]
equations = [f"Equation {i+1}" for i in range(len(RAW_EQS))]

columns = [f"{eq}_{metric}" for eq in equations for metric in metrics]

summary_df = pd.DataFrame(
    columns=["Storm Index"] + columns,
)

for storm_index, storm in enumerate(storms):
    summary_df.loc[storm_indices[storm_index], "Storm Index"] = storm_indices[storm_index]


for storm_index, storm in enumerate(storms):
    y_true = storm["Observed_DST"].values
    
    for eq_index in range(len(RAW_EQS)):
    
        res_eq = storm[f"Pred_DST_Equation_{eq_index+1}"].values
    

        m_eq = baseline_models.get_all_metrics_dict(y_true, res_eq)
        
        for metric in metrics:
            summary_df.loc[storm_indices[storm_index], f"Equation {eq_index+1}_{metric}"] = m_eq[metric]
        
        

summary_df.loc[len(summary_df)] = ["Mean", *summary_df[columns].mean().values]


global_data = pd.concat(storms, ignore_index=True)
y_true = global_data["Observed_DST"].values

summary_df.loc[len(summary_df), "Storm Index"] = 'Global'

for eq_index in range(len(RAW_EQS)):
    res_eq = global_data[f"Pred_DST_Equation_{eq_index+1}"].values
    m_eq = baseline_models.get_all_metrics_dict(y_true, res_eq)    
    for metric in metrics:
        summary_df.loc[len(summary_df) - 1, f"Equation {eq_index+1}_{metric}"] = m_eq[metric]


display(summary_df)

,Storm Index,Equation 1_RMSE,Equation 1_MAE,Equation 1_R2,Equation 1_CC,Equation 1_BFE,Equation 2_RMSE,Equation 2_MAE,Equation 2_R2,Equation 2_CC,...,Equation 6_RMSE,Equation 6_MAE,Equation 6_R2,Equation 6_CC,Equation 6_BFE,Equation 7_RMSE,Equation 7_MAE,Equation 7_R2,Equation 7_CC,Equation 7_BFE
54,54,9.745655,6.83855,0.765586,0.923463,12.618396,9.170363,6.306532,0.792444,0.929433,...,9.950981,6.892319,0.755605,0.924818,13.239107,8.673717,6.588866,0.814317,0.928641,11.301985
55,55,15.45666,12.125151,0.802554,0.914472,17.293221,15.477186,12.047038,0.802029,0.914773,...,15.516899,12.059337,0.801012,0.913815,17.1403,16.449289,12.926959,0.77638,0.900502,18.164656
56,56,11.923871,9.441087,0.687895,0.864996,13.776978,11.76299,9.26387,0.69626,0.871547,...,11.595705,9.033693,0.704838,0.869104,13.571008,11.831529,9.30636,0.69271,0.883821,14.710407
57,57,10.548566,8.06729,0.717111,0.885463,10.447585,10.641121,8.014319,0.712125,0.893614,...,10.129202,7.717149,0.739156,0.896014,9.454292,9.834485,7.633421,0.754114,0.906933,9.969161
58,58,9.316351,6.358408,0.790282,0.915747,15.773875,8.454199,5.979033,0.827301,0.919482,...,8.737426,5.972947,0.815536,0.920243,15.027077,8.344028,5.728047,0.831773,0.931581,14.176383
59,59,19.212159,16.20118,0.628309,0.910652,12.301995,18.93238,16.047103,0.639056,0.921116,...,18.755635,15.918559,0.645763,0.916767,12.317915,15.741868,13.450539,0.750459,0.94344,10.585746
60,60,16.652461,13.110027,0.801415,0.960356,20.074287,16.513636,13.513491,0.804712,0.958955,...,15.905531,13.17985,0.81883,0.963424,17.604023,15.842252,12.058539,0.820269,0.957678,20.299388
61,61,11.789867,8.40107,0.917132,0.961177,17.16978,11.88397,8.886693,0.915804,0.96213,...,12.410941,8.671901,0.908172,0.95925,18.248169,10.534158,7.737233,0.933844,0.966963,16.059308
62,62,12.65854,10.555086,0.627468,0.801518,14.578574,12.74353,10.836518,0.622448,0.810219,...,12.239675,10.29346,0.651714,0.826605,13.171476,11.981492,9.955988,0.666252,0.842861,12.976021
63,63,13.332099,8.723691,0.870862,0.962578,20.154227,12.608102,8.299471,0.884507,0.961065,...,12.509199,8.250986,0.886312,0.965227,18.605889,11.964242,8.226228,0.896001,0.961207,16.950241


In [ ]:
print(summary_df.to_latex(index=False, float_format="%.3f").replace("_", " ").replace("Equation ", "Eq ").replace("Storm Index", "Storm"))

\begin{tabular}{llllllllllllllllllllllllllllllllllll}
\toprule
Storm & Eq 1 RMSE & Eq 1 MAE & Eq 1 R2 & Eq 1 CC & Eq 1 BFE & Eq 2 RMSE & Eq 2 MAE & Eq 2 R2 & Eq 2 CC & Eq 2 BFE & Eq 3 RMSE & Eq 3 MAE & Eq 3 R2 & Eq 3 CC & Eq 3 BFE & Eq 4 RMSE & Eq 4 MAE & Eq 4 R2 & Eq 4 CC & Eq 4 BFE & Eq 5 RMSE & Eq 5 MAE & Eq 5 R2 & Eq 5 CC & Eq 5 BFE & Eq 6 RMSE & Eq 6 MAE & Eq 6 R2 & Eq 6 CC & Eq 6 BFE & Eq 7 RMSE & Eq 7 MAE & Eq 7 R2 & Eq 7 CC & Eq 7 BFE \\
\midrule
54 & 9.746 & 6.839 & 0.766 & 0.923 & 12.618 & 9.170 & 6.307 & 0.792 & 0.929 & 12.136 & 9.856 & 6.843 & 0.760 & 0.925 & 13.042 & 8.695 & 6.595 & 0.813 & 0.929 & 11.029 & 9.039 & 6.153 & 0.798 & 0.930 & 12.028 & 9.951 & 6.892 & 0.756 & 0.925 & 13.239 & 8.674 & 6.589 & 0.814 & 0.929 & 11.302 \\
55 & 15.457 & 12.125 & 0.803 & 0.914 & 17.293 & 15.477 & 12.047 & 0.802 & 0.915 & 16.816 & 15.441 & 12.018 & 0.803 & 0.915 & 16.969 & 16.373 & 12.792 & 0.778 & 0.903 & 18.150 & 15.514 & 12.034 & 0.801 & 0.915 & 16.761 & 15.517 & 12.059 & 0.801 & 0.

In [ ]:
display(summary_df[['Storm Index', 'Equation 1_BFE', 'Equation 2_BFE', 'Equation 3_BFE']].set_index('Storm Index'))

,Equation 1_BFE,Equation 2_BFE,Equation 3_BFE
Storm Index,,,
54,12.618396,12.136333,13.041554
55,17.293221,16.815952,16.968631
56,13.776978,13.86179,13.684032
57,10.447585,9.156381,9.319015
58,15.773875,14.343868,14.974158
59,12.301995,11.732671,12.257052
60,20.074287,18.458648,17.448155
61,17.16978,17.415894,18.06918
62,14.578574,13.080866,13.215677


## Train storms

In [ ]:
storms = []
storm_indices = []
models = {}

for RAW_EQ in RAW_EQS:
    model = EquationModel(RAW_EQ, FEATURES, is_template=MODE == "template")
    models[RAW_EQ] = model
    
    
test_storms = storm_dates.TRAIN_STORMS_SYMBOLIC_REGRESSION

for sd, ed, storm_id in tqdm(test_storms):
    start = pd.to_datetime(sd)
    end = pd.to_datetime(ed)
    storm_df = data[
        start - pd.DateOffset(hours=1) : end + pd.DateOffset(hours=1)
    ].copy()
    if storm_df.empty:
        continue

    file_name = f"storm_{storm_id}.png"
    predict_and_plot_storm(
        models,
        RAW_EQS,
        start,
        end,
        storm_df,
        storm_id,
        os.path.join(OUTPUT_DIR, file_name),
    )

    csv_name = f"data_storm_{storm_id}.csv"
    storms.append(
        save_prediction_data(
            models, RAW_EQS, start, end, storm_df, os.path.join(OUTPUT_DIR, csv_name)
        )
    )
    storm_indices.append(storm_id)


  0%|          | 0/53 [00:00<?, ?it/s]

100%|██████████| 53/53 [00:44<00:00,  1.19it/s]


In [ ]:
metrics = ["RMSE", "MAE", "R2", "CC", "BFE"]
equations = [f"Equation {i+1}" for i in range(len(RAW_EQS))]

columns = [f"{eq}_{metric}" for eq in equations for metric in metrics]

summary_df = pd.DataFrame(
    columns=["Storm Index"] + columns,
)

for storm_index, storm in enumerate(storms):
    summary_df.loc[len(summary_df), "Storm Index"] = storm_indices[storm_index]


for storm_index, storm in enumerate(storms):
    y_true = storm["Observed_DST"].values
    
    for eq_index in range(len(RAW_EQS)):
    
        res_eq = storm[f"Pred_DST_Equation_{eq_index+1}"].values
    

        m_eq = baseline_models.get_all_metrics_dict(y_true, res_eq)
        
        for metric in metrics:
            summary_df.loc[storm_indices[storm_index], f"Equation {eq_index+1}_{metric}"] = m_eq[metric]
        
        

display(summary_df.mean())

summary_df.loc[len(summary_df)] = ["Mean", *summary_df[columns].mean().values]

global_data = pd.concat(storms, ignore_index=True)
y_true = global_data["Observed_DST"].values

summary_df.loc[len(summary_df), "Storm Index"] = 'Global'



for eq_index in range(len(RAW_EQS)):
    res_eq = global_data[f"Pred_DST_Equation_{eq_index+1}"].values
    m_eq = baseline_models.get_all_metrics_dict(y_true, res_eq)    
    for metric in metrics:
        summary_df.loc[len(summary_df) - 1, f"Equation {eq_index+1}_{metric}"] = m_eq[metric]


display(summary_df)

Storm Index             27.0
Equation 1_RMSE     14.79831
Equation 1_MAE     11.361231
Equation 1_R2       0.716058
Equation 1_CC       0.903088
Equation 1_BFE     17.548806
Equation 2_RMSE    15.140747
Equation 2_MAE     11.611798
Equation 2_R2       0.695146
Equation 2_CC       0.901443
Equation 2_BFE     17.681434
Equation 3_RMSE    14.994499
Equation 3_MAE     11.420535
Equation 3_R2       0.705673
Equation 3_CC       0.903823
Equation 3_BFE      17.66812
Equation 4_RMSE    14.903314
Equation 4_MAE     11.412277
Equation 4_R2       0.689535
Equation 4_CC       0.907816
Equation 4_BFE     16.511012
Equation 5_RMSE    15.236143
Equation 5_MAE     11.679358
Equation 5_R2       0.689152
Equation 5_CC        0.90072
Equation 5_BFE     17.731908
Equation 6_RMSE    14.998827
Equation 6_MAE     11.411055
Equation 6_R2       0.704817
Equation 6_CC       0.903443
Equation 6_BFE      17.71539
Equation 7_RMSE    15.135002
Equation 7_MAE     11.574196
Equation 7_R2       0.675934
Equation 7_CC 

,Storm Index,Equation 1_RMSE,Equation 1_MAE,Equation 1_R2,Equation 1_CC,Equation 1_BFE,Equation 2_RMSE,Equation 2_MAE,Equation 2_R2,Equation 2_CC,...,Equation 6_RMSE,Equation 6_MAE,Equation 6_R2,Equation 6_CC,Equation 6_BFE,Equation 7_RMSE,Equation 7_MAE,Equation 7_R2,Equation 7_CC,Equation 7_BFE
0,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2,16.69703,14.045504,0.754609,0.895863,14.264534,16.757259,13.982998,0.752835,0.888442,...,16.086924,13.570816,0.772214,0.903459,12.887402,18.584191,15.525301,0.696004,0.874794,12.086484
2,3,18.563751,14.425571,0.115975,0.602333,18.089756,19.050206,14.731976,0.069037,0.612635,...,18.801806,14.500514,0.093157,0.613738,17.878054,21.427087,16.420962,-0.177767,0.592612,18.282762
3,4,11.467075,9.162138,0.873586,0.935606,12.277404,12.181117,9.550641,0.857352,0.931072,...,11.293233,8.908971,0.87739,0.939516,11.954189,11.914841,9.455888,0.863521,0.93696,10.758006
4,5,15.181822,12.164943,0.827508,0.922936,19.47891,15.825761,12.424307,0.812565,0.920574,...,15.765761,12.461815,0.813984,0.918462,20.555896,16.437194,13.257719,0.797803,0.920208,19.473229
5,6,18.57923,13.316745,0.769047,0.883514,27.046916,18.543568,13.511804,0.769933,0.885451,...,18.662434,13.279726,0.766974,0.881933,27.149181,18.091928,13.923127,0.781003,0.898138,24.875869
6,7,14.606604,10.395774,0.686383,0.914837,17.7651,16.45387,11.524124,0.602042,0.909021,...,16.81834,11.590346,0.584216,0.908764,20.755919,17.857359,12.964298,0.531256,0.882932,21.262446
7,8,10.557127,8.864373,0.805295,0.904136,8.777854,10.609987,8.71615,0.80334,0.901667,...,10.314539,8.494894,0.81414,0.906417,7.891186,10.368487,8.432769,0.812191,0.910651,6.395113
8,9,11.994192,9.861083,0.853669,0.945525,11.657361,11.936519,9.643098,0.855072,0.947607,...,11.556549,9.363383,0.864152,0.948941,11.449887,12.099101,9.407002,0.851098,0.954657,12.729879
9,10,11.041145,7.63761,0.763208,0.919024,9.339973,12.221125,8.251979,0.709891,0.919724,...,11.65861,7.944287,0.735983,0.917485,9.840167,13.265519,9.335378,0.658188,0.917989,11.860631


In [ ]:
display(summary_df[['Storm Index', 'Equation 1_BFE', 'Equation 2_BFE', 'Equation 3_BFE']].set_index('Storm Index'))

,Equation 1_BFE,Equation 2_BFE,Equation 3_BFE
Storm Index,,,
1,NaN,NaN,NaN
2,14.264534,13.787673,12.802373
3,18.089756,17.540217,17.71867
4,12.277404,13.113295,12.283887
5,19.47891,20.133902,20.383505
6,27.046916,26.864577,27.039299
7,17.7651,20.157141,20.76359
8,8.777854,7.42574,7.941341
9,11.657361,12.044789,11.498306


In [ ]:

storms = range(54, 74)
parent_folder = 'template-primitive-figures'

for storm_number in storms:
    # We use f-strings with double {{ }} to escape the LaTeX braces
    # and single { } for the Python variables.
    latex_code = f"""
\\begin{{figure}}[ht]
    \\centering
    \\includegraphics[width=\\textwidth]{{{parent_folder}/storm_{storm_number}.png}}
    \\caption{{Reconstruction of storm {storm_number} using the Equations generated from the templated symbolic regression with the primitive features}}\\label{{fig:template-primitive-storm-{storm_number}}}
\\end{{figure}}
"""
    print(latex_code)   


\begin{figure}[ht]
    \centering
    \includegraphics[width=\textwidth]{template-primitive-figures/storm_54.png}
    \caption{Reconstruction of storm 54 using the Equations generated from the templated symbolic regression with the primitive features}\label{fig:template-primitive-storm-54}
\end{figure}


\begin{figure}[ht]
    \centering
    \includegraphics[width=\textwidth]{template-primitive-figures/storm_55.png}
    \caption{Reconstruction of storm 55 using the Equations generated from the templated symbolic regression with the primitive features}\label{fig:template-primitive-storm-55}
\end{figure}


\begin{figure}[ht]
    \centering
    \includegraphics[width=\textwidth]{template-primitive-figures/storm_56.png}
    \caption{Reconstruction of storm 56 using the Equations generated from the templated symbolic regression with the primitive features}\label{fig:template-primitive-storm-56}
\end{figure}


\begin{figure}[ht]
    \centering
    \includegraphics[width=\textwidth]{template